# SoundStream Demo

This notebook clones soundstream project, installs dependencies, downloads model weights from HuggingFace and runs the model on audio.

## Settings

You can change AUDIO_URl to yours.

In [1]:
REPO_URL = "https://github.com/DoubleM26/SoundStream"
HF_REPO_ID = "mishgun100/soundstream"
CHECKPOINT_NAME = "final_train.pt"
AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"
device = "cpu"
model_sr = 16000

## Clone and Install

In [2]:
!rm -rf soundstream
!git clone {REPO_URL} soundstream
%cd soundstream
!pip install -q -r requirements.txt

Cloning into 'soundstream'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 74 (delta 32), reused 74 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 15.41 KiB | 2.57 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/soundstream
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.0/787.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.2 MB/s eta 0:00:00


## Model load

In [3]:
import torch
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(repo_id=HF_REPO_ID, filename=CHECKPOINT_NAME)
checkpoint = torch.load(checkpoint_path, map_location=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


final_train.pt:   0%|          | 0.00/377M [00:00<?, ?B/s]

In [4]:
from src.model.model import SoundStream
model = SoundStream(checkpoint["config"]).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()

SoundStream(
  (encoder): Encoder(
    (net): Sequential(
      (0): CausalConv(1, 32, kernel_size=(7,), stride=(1,))
      (1): EncoderBlock(
        (net): Sequential(
          (0): ResidualUnit(
            (net): Sequential(
              (0): CausalConv(32, 32, kernel_size=(7,), stride=(1,))
              (1): ELU(alpha=1.0)
              (2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
            )
          )
          (1): ResidualUnit(
            (net): Sequential(
              (0): CausalConv(32, 32, kernel_size=(7,), stride=(1,), dilation=(3,))
              (1): ELU(alpha=1.0)
              (2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
            )
          )
          (2): ResidualUnit(
            (net): Sequential(
              (0): CausalConv(32, 32, kernel_size=(7,), stride=(1,), dilation=(9,))
              (1): ELU(alpha=1.0)
              (2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
            )
          )
          (3): CausalConv(32, 64, kern

## Original Audio

In [5]:
from IPython.display import Audio
import requests
import torchaudio

original_path = "demo_original.wav"
with open(original_path, "wb") as f:
    f.write(requests.get(AUDIO_URL).content)

audio, sr = torchaudio.load(original_path)
Audio(audio.numpy(), rate=sr)

## Reconstructed Audio

In [6]:
recon_path = "demo_recon.wav"
audio2 = torchaudio.functional.resample(audio, sr, model_sr)
with torch.no_grad():
    recon = model(audio2.unsqueeze(0).to(device))["recon"][0].cpu().clamp(-1, 1)
torchaudio.save(recon_path, recon, model_sr)
Audio(recon_path)